In [2]:
! which python

/Users/kunal21/opt/miniconda3/envs/v_agentic_kn/bin/python


### Tools in LangChain

- It’s like a well-structured instruction manual for your AI assistant to perform a specific task, like fetching current rates from some stocks portal
- All related atributes can be found in BaseTool interface, can be called as @Tool

In [3]:
from langchain_core.tools import tool

In [4]:
@tool
def rates_tool(price:float , discount_percentage : float) -> float :
    """
    Calculates final price after applying discount

    Args :
        price(float) - original price of item
        discount_percentage(float) - discount applicable 

    Returns : 
        Final price after applying discount(float)

    """
    if not (0 < discount_percentage <100):
        raise ValueError('discount percentage not valid')
    
    return price - (price * (discount_percentage/100))


In [5]:
rates_tool.name, rates_tool.description, rates_tool.args

('rates_tool',
 'Calculates final price after applying discount\n\nArgs :\n    price(float) - original price of item\n    discount_percentage(float) - discount applicable \n\nReturns : \n    Final price after applying discount(float)',
 {'price': {'title': 'Price', 'type': 'number'},
  'discount_percentage': {'title': 'Discount Percentage', 'type': 'number'}})

In [6]:
rates_tool.invoke({'price':100.0,'discount_percentage':5.5})

94.5

In [40]:
@tool
def employee_finder(emp_name:str ) -> str :
    """
    searches employee name in JPMC employee list and returns true or false

    Args :
        emp_name(str) - name of JPMC employee

    Returns : 
        If employee exists in company JPMC

    """
    JPMC = ['KK','Kunal','Omi']
    return "FOUND" if (emp_name in JPMC) else "NOT FOUND !"


In [41]:
employee_finder.invoke({'emp_name':'Omi'})

'FOUND'

### Binding Tools to model

In [42]:
import dotenv
import os

dotenv.load_dotenv()
GROQ_KEY = os.getenv('GROQ_KEY')


In [43]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama3-8b-8192', #'deepseek-r1-distill-llama-70b',#llama-3.3-70b-versatile
               api_key=GROQ_KEY)


In [44]:
llm.bind_tools([rates_tool,employee_finder])

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x134fc4690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x134fc5d10>, model_name='llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'rates_tool', 'description': 'Calculates final price after applying discount\n\nArgs :\n    price(float) - original price of item\n    discount_percentage(float) - discount applicable \n\nReturns : \n    Final price after applying discount(float)', 'parameters': {'properties': {'price': {'type': 'number'}, 'discount_percentage': {'type': 'number'}}, 'required': ['price', 'discount_percentage'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'employee_finder', 'description': 'searches employee name in JPMC employee list and returns true or false\n\nArgs :\n    emp_name(str) - name of JPMC employee\n\nReturns : \n    If employee exists in 

In [45]:
## llm will use tool if relevant prompt given

response = llm.invoke('Fetch rates for an item that costs $100 after a 20% discount?')
response.content

'To find the discount rate, we can convert the percentage to a decimal by dividing by 100:\n\n20% = 20/100 = 0.20\n\nThe discount is 20% of the original price, which is:\n\n$100 x 0.20 = $20\n\nSo, the discount is $20.\n\nTo find the new price after the discount, we subtract the discount from the original price:\n\n$100 - $20 = $80\n\nTherefore, the price after the 20% discount is $80.'

In [46]:
response.tool_calls

[]

In [51]:
llm_with_tools = llm.bind_tools([employee_finder,rates_tool])
response = llm_with_tools.invoke('check if emp_name Omi is in JPMC employee list')

print(response,'\n')
print(response.content)         # For the main response text
print(response.tool_calls)      # To see tool call details (if available)

content='' additional_kwargs={'tool_calls': [{'id': 'kvaa3mja9', 'function': {'arguments': '{"emp_name":"Omi"}', 'name': 'employee_finder'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 1084, 'total_tokens': 1159, 'completion_time': 0.064934116, 'prompt_time': 0.121673764, 'queue_time': 0.26520525699999997, 'total_time': 0.18660788}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_5b339000ab', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--1579e972-e7ce-4e27-b8ae-65e2ebd75161-0' tool_calls=[{'name': 'employee_finder', 'args': {'emp_name': 'Omi'}, 'id': 'kvaa3mja9', 'type': 'tool_call'}] usage_metadata={'input_tokens': 1084, 'output_tokens': 75, 'total_tokens': 1159} 


[{'name': 'employee_finder', 'args': {'emp_name': 'Omi'}, 'id': 'kvaa3mja9', 'type': 'tool_call'}]


In [50]:
response.content

''

In [61]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool

llm = ChatGroq(model='llama-3.1-8b-instant', #'deepseek-r1-distill-llama-70b',#llama-3.3-70b-versatile
               api_key=GROQ_KEY)
@tool
def calculate_discount(price: float, discount_percentage: float) -> float:
    """
    Calculates the final price after applying a discount.

    Args:
        price (float): The original price of the item.
        discount_percentage (float): The discount percentage (e.g., 20 for 20%).

    Returns:
        float: The final price after the discount is applied.
    """
    if not (0 <= discount_percentage <= 100):
        raise ValueError("Discount percentage must be between 0 and 100")

    discount_amount = price * (discount_percentage / 100)
    final_price = price - discount_amount
    return final_price
    
llm_with_tools = llm.bind_tools([calculate_discount,employee_finder])


result = llm_with_tools.invoke("What is the price of an item that costs $100 after a 20% discount?")
print(result.tool_calls)

args = result.tool_calls[0]['args']
print(calculate_discount.invoke(args), result.content)

[{'name': 'calculate_discount', 'args': {'discount_percentage': 20, 'price': 100}, 'id': 'hxqy6d3s1', 'type': 'tool_call'}]
80.0 


In [58]:
result = llm_with_tools.invoke("check if employee Omi is in JPMC")
print(result.tool_calls)
print(result.content)

[{'name': 'employee_finder', 'args': {'emp_name': 'Omi'}, 'id': 'baxdje7qt', 'type': 'tool_call'}]



In [60]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool

llm = ChatGroq(model='llama-3.1-8b-instant', #'deepseek-r1-distill-llama-70b',#llama-3.3-70b-versatile
               api_key=GROQ_KEY)

@tool
def employee_finder(emp_name:str ) -> str :
    """
    searches employee name in JPMC employee list and returns true or false

    Args :
        emp_name(str) - name of JPMC employee

    Returns : 
        If employee exists in company JPMC

    """
    JPMC = ['KK','Kunal','Omi']
    return "FOUND" if (emp_name in JPMC) else "NOT FOUND !"
    
llm_with_tools = llm.bind_tools([employee_finder])


result = llm_with_tools.invoke("Check if employee Kunal is in JPMC")
print(result.tool_calls)

args = result.tool_calls[0]['args']
print(employee_finder.invoke(args))

[{'name': 'employee_finder', 'args': {'emp_name': 'Kunal'}, 'id': '0t6atknt5', 'type': 'tool_call'}]
FOUND


In [66]:
###
# Why is result.content empty?
# When you bind tools to a chat model in LangChain and invoke it with a question that should trigger the tool, the model’s main response (i.e., content) is typically blank because it “stops” at the tool call instruction.
# Instead, the model’s primary output is a list of tool calls (in result.tool_calls). The actual content (LLM-generated text) is suppressed when the model decides to use a tool, and you have to process and stitch together the tool results yourself.
# In other words:
# result.tool_calls: Contains details about which tools the model wants to call and their arguments.
# result.content: Will usually be empty if a tool is called, because the model expects a loop where you call the tool, get its output, and send it back for a final answer.
# How to Fix: Make the LLM Return Useful Content
# To get a final answer with the tool’s result (e.g., "FOUND" or "NOT FOUND !"), you need to invoke the tool, get its output, and pass that output back to the LLM for a final response.
###

# Step 1: Check for tool calls

from langchain_groq import ChatGroq
from langchain_core.tools import tool

llm = ChatGroq(model='llama-3.1-8b-instant', #'deepseek-r1-distill-llama-70b',#llama-3.3-70b-versatile
               api_key=GROQ_KEY)

@tool
def employee_finder(emp_name:str ) -> str :
    """
    searches employee name in JPMC employee list and returns true or false

    Args :
        emp_name(str) - name of JPMC employee

    Returns : 
        If employee exists in company JPMC

    """
    JPMC = ['KK','Kunal','Omi']
    return "FOUND" if (emp_name in JPMC) else "NOT FOUND !"
    
llm_with_tools = llm.bind_tools([employee_finder])


result = llm_with_tools.invoke("Check if employee Kunal is in JPMC")
print(result.tool_calls)



# args = result.tool_calls[0]['args']
# print(employee_finder.invoke(args))


if result.tool_calls:
    tool_name = result.tool_calls[0]["name"]
    arguments = result.tool_calls[0]["args"]
    
    # Step 2: Call the tool with its arguments (in this case, employee_finder)
    tool_output = employee_finder.invoke(arguments)
    
    # Step 3: Send the tool result back to the LLM for a final answer
    result = llm.invoke(f"Answer the original question based on this tool output:\n{tool_output}")
    
    print("Final answer:", result.content)
else:
    # If no tool calls, just print the LLM's response
    print("LLM's direct answer:", result.content)

[{'name': 'employee_finder', 'args': {'emp_name': 'Kunal'}, 'id': '1s1hz2abc', 'type': 'tool_call'}]
Final answer: It seems like the tool output is "FOUND". However, I don't have enough context to provide a meaningful answer. Can you please provide more information or clarify what you are trying to ask?
